# CUM Series 12 — Bifurcation Sweep, TD(lambda), Adaptive Scheduling

Deep research v3 suggested directions:
1. **12a**: Sweep |p'(sigma*)| to find optimal oscillation strength
2. **12b**: TD(lambda) exponentially-weighted blending of ALL NS iterates
3. **12c**: Adaptive oscillation scheduling (anneal polynomial over training)

Baseline: Combined mode = -0.016 ± 0.004 vs Muon (3 runs at 1.2M, holds at 124M)

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import CUM8v1, CUM12v1, CUM12v2, CUM12v3
from cum.cum_12v1 import bifurcation_coeffs

# Verify coefficient derivation
for d in [-1.0, -1.4, -1.58, -2.0, -2.8]:
    a, b, c = bifurcation_coeffs(d)
    print(f"p'={d:+.2f}: a={a:.4f}, b={b:.4f}, c={c:.4f}")
print('Imports OK')

In [ ]:
MODEL_CFG = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
print(f'Model: {sum(p.numel() for p in test_model.parameters())/1e6:.1f}M params')
del test_model

In [ ]:
class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y

dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    t = cfg['type']

    if t == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
    elif t == '8v1':
        main_opt = CUM8v1(
            hidden_2d, lr=BASE_LR,
            method=cfg.get('method', 'matrix'),
            mode=cfg.get('mode', 'combined'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
            total_steps=TOTAL_STEPS,
        )
    elif t == '12v1':
        main_opt = CUM12v1(
            hidden_2d, lr=BASE_LR,
            deriv=cfg.get('deriv', -1.58),
            mode=cfg.get('mode', 'combined'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
        )
    elif t == '12v2':
        main_opt = CUM12v2(
            hidden_2d, lr=BASE_LR,
            td_lambda=cfg.get('td_lambda', 0.5),
            ns_steps=cfg.get('ns_steps', 5),
            use_temporal=cfg.get('use_temporal', True),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
        )
    elif t == '12v3':
        main_opt = CUM12v3(
            hidden_2d, lr=BASE_LR,
            deriv_start=cfg.get('deriv_start', -1.8),
            deriv_end=cfg.get('deriv_end', -1.0),
            total_steps=TOTAL_STEPS,
            mode=cfg.get('mode', 'combined'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main_opt, aux_opt


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, main_opt, aux_opt, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='12a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    combined = next((r['val_loss'] for r in results
                     if r.get('name', '').startswith('8v1 combined')), None)

    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon | vs Combined | Time |')
    print(f'|--------|----------|---------|-------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | -- | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '--'
        vc = f'{r["val_loss"] - combined:+.4f}' if combined else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {vc} | {r["elapsed"]:.0f}s |')

    new = [r for r in sorted(results, key=lambda x: x['val_loss'])
           if r['type'] not in ('Muon',) and not r.get('error')
           and not r.get('name', '').startswith('8v1 combined')]
    if new:
        b = new[0]
        print(f'\nBEST NEW: {b["name"]} -> {b["val_loss"]:.4f}', end='')
        if muon: print(f' (vs Muon: {b["val_loss"]-muon:+.4f})', end='')
        if combined: print(f' (vs Combined: {b["val_loss"]-combined:+.4f})', end='')
        print()


print('Factory ready')

---
## 12a: Bifurcation Diagram Sweep (Batch 1)

Sweep p'(sigma*) from -1.0 (weakly oscillating) through -1.4 (moderate).
Theory predicts inverted-U with peak in [1.2, 1.8] magnitude.
Standard Muon is at -1.58.

In [ ]:
CONFIGS_12A = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined (ref)', 'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    {'name': '12v1 d=-1.0 combined', 'type': '12v1', 'deriv': -1.0, 'mode': 'combined'},
    {'name': '12v1 d=-1.4 combined', 'type': '12v1', 'deriv': -1.4, 'mode': 'combined'},
]

results_12a = run_all(CONFIGS_12A)
show_results(results_12a, '12a: Bifurcation Batch 1')

---
## 12a2: Bifurcation Diagram Sweep (Batch 2)

Test stronger oscillation: -2.0, -2.8.
Also -1.58 as sanity check (should match 8v1 combined).

In [ ]:
CONFIGS_12A2 = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '12v1 d=-1.58 combined', 'type': '12v1', 'deriv': -1.58, 'mode': 'combined'},
    {'name': '12v1 d=-2.0 combined', 'type': '12v1', 'deriv': -2.0, 'mode': 'combined'},
    {'name': '12v1 d=-2.8 combined', 'type': '12v1', 'deriv': -2.8, 'mode': 'combined'},
]

results_12a2 = run_all(CONFIGS_12A2)
show_results(results_12a2, '12a2: Bifurcation Batch 2')

---
## 12b: TD(lambda) Sweep

Exponentially weight all NS iterates.
lambda=0 is pure Muon, lambda=1 is uniform average.

In [ ]:
CONFIGS_12B = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined (ref)', 'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    {'name': '12v2 lam=0.3 +temporal', 'type': '12v2', 'td_lambda': 0.3, 'use_temporal': True},
    {'name': '12v2 lam=0.5 +temporal', 'type': '12v2', 'td_lambda': 0.5, 'use_temporal': True},
]

results_12b = run_all(CONFIGS_12B)
show_results(results_12b, '12b: TD(lambda) Batch 1')

---
## 12b2: TD(lambda) Extended

Higher lambda values + ablation (TD without temporal EMA).

In [ ]:
CONFIGS_12B2 = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '12v2 lam=0.7 +temporal', 'type': '12v2', 'td_lambda': 0.7, 'use_temporal': True},
    {'name': '12v2 lam=0.9 +temporal', 'type': '12v2', 'td_lambda': 0.9, 'use_temporal': True},
    {'name': '12v2 lam=0.5 no-temporal', 'type': '12v2', 'td_lambda': 0.5, 'use_temporal': False},
]

results_12b2 = run_all(CONFIGS_12B2)
show_results(results_12b2, '12b2: TD(lambda) Batch 2')

---
## 12c: Adaptive Oscillation Scheduling

Anneal polynomial coefficients: strong oscillation early -> weak late.
Like LR scheduling but for optimizer dynamics.

In [ ]:
CONFIGS_12C = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': '8v1 combined (ref)', 'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    {'name': '12v3 -1.8 -> -1.0', 'type': '12v3', 'deriv_start': -1.8, 'deriv_end': -1.0},
    {'name': '12v3 -2.2 -> -1.2', 'type': '12v3', 'deriv_start': -2.2, 'deriv_end': -1.2},
]

results_12c = run_all(CONFIGS_12C)
show_results(results_12c, '12c: Adaptive Scheduling')